# Kaggle Netflow Graph Dataset: Construction and Visualization

This notebook covers **only the pre-built Kaggle graph dataset** (`data/kaggle/`).
The companion notebook, `02_graph_construction_and_visualization_ids_2017.ipynb`,
covers the CIC-IDS-2017 raw flow logs separately. Splitting them keeps each
notebook focused on one data source, so a reader is never unsure which
dataset a given plot or number came from.

**What this dataset is.** Three GraphML files, all covering the same
41,073 IP addresses, built from 100,000 sampled NetFlow-style records
(the "0.1M" in the filenames). Each edge carries 7 z-scored (standardized)
numeric features - packet count, byte counts, duration, protocol, direction,
connection state - plus a ground-truth `ActivityLabel` (0 = benign,
1 = malicious).

**What this notebook answers, precisely enough to present and defend to the
class:**
1. What exactly is different between the "plain", "multi", and "aggregated"
   files - with real numbers, not assumptions.
2. What does this graph's structure actually look like, and why (degree
   distribution, hub dominance, connected components)?
3. Where do the attacks sit structurally - on the busiest nodes, or
   elsewhere?
4. How do the standard centrality features behave on a graph shaped like
   this one?

Every claim below is backed by a computed number in the cell above it -
if you re-run this notebook, you should get the same numbers.


In [13]:
%matplotlib inline

import sys
sys.path.append("../src")

import networkx as nx
import pandas as pd
from collections import Counter

from load_kaggle_graph import load_graphml, mark_malicious_nodes, summarize_graph, compare_graph_variants
from graph_features import compute_all_features, verify_features
from visualize import (
    plot_graph, plot_multiple_graphs, plot_degree_distribution,
    get_readable_subgraph, get_subgraph_excluding_nodes,
)

pd.set_option("display.max_columns", 12)

KAGGLE_DIR = "../data/kaggle"
VARIANTS = {
    "plain":      f"{KAGGLE_DIR}/0.1M-Stratified.graphml",
    "multi":      f"{KAGGLE_DIR}/0.1M-Stratified-Multi.graphml",
    "aggregated": f"{KAGGLE_DIR}/0.1M-Stratified-aggregated.graphml",
}


ModuleNotFoundError: No module named 'networkx'

## 1. Loading all three variants and comparing them directly

Each file is on the order of 20-40MB (~41K nodes) so all three fit in
memory comfortably - no sampling needed for this step.


In [7]:
comparison = compare_graph_variants(VARIANTS)
comparison


NameError: name 'compare_graph_variants' is not defined

**Reading this table:** "plain" and "aggregated" have identical node and
edge counts (45,221 edges - one per unique IP pair). "multi" has more than
double that (100,000 edges - the ActivityLabel/multigraph. It's the only one that keeps
every individual flow as its own parallel edge, matching the "0.1M" =
100,000 original flow records exactly.


In [ ]:
graphs = {name: mark_malicious_nodes(load_graphml(path)) for name, path in VARIANTS.items()}
G, G_multi, G_agg = graphs["plain"], graphs["multi"], graphs["aggregated"]

for name, G_ in graphs.items():
    print(name, "->", summarize_graph(G_, name=name))


## 2. What "plain" vs "aggregated" actually means, with real numbers

Both files store exactly one edge per unique (source, destination) pair.
The question is: when two IPs exchanged more than one flow, what value
ends up on that single edge?

We pick a real pair from this dataset that had **16** separate flows
between them in the "multi" file, and compare all three representations
directly.


In [ ]:
hub = max(G.degree, key=lambda x: x[1])[0]
print("Busiest node in this graph:", hub, "- degree:", G.degree(hub))

# find a neighbor of the hub that had several parallel flows in the multi-graph
parallel_counts = Counter()
for u, v, k in G_multi.edges(hub, keys=True):
    other = v if u == hub else u
    parallel_counts[other] += 1
busy_neighbor, n_parallel = parallel_counts.most_common(1)[0]
print(f"Neighbor with the most repeated flows to the hub: {busy_neighbor} ({n_parallel} flows)")

plain_edge = dict(G[hub][busy_neighbor])
agg_edge = dict(G_agg[hub][busy_neighbor])
multi_values = [float(d.get("TotPkts")) for d in G_multi[hub][busy_neighbor].values()]

print("\nPLAIN edge TotPkts      :", plain_edge.get("TotPkts"))
print("AGGREGATED edge TotPkts :", agg_edge.get("TotPkts"))
print(f"\nAcross the {n_parallel} individual MULTI flows for this pair, TotPkts values were:")
print([round(v, 6) for v in multi_values])
print("Mean of those values   :", sum(multi_values) / len(multi_values))


**Precise conclusion - don't overstate this:** "plain" and "aggregated"
give *different* values for pairs with more than one underlying flow (confirmed
above), so they are genuinely different representations, not duplicates.
However, the "aggregated" value is **not** simply the mean, median, or sum
of the individual flow values (check the numbers above - none of those
match exactly). The dataset does not document its exact aggregation
formula, so the honest statement for a presentation is: *"aggregated"
combines repeated flows between a pair into one edge using an unspecified
aggregation rule that is not a simple average* - not "aggregated = summed"
as one might assume from the name alone. When in doubt, say what you can
verify, not what seems intuitive.


## 3. Degree distribution: this graph is hub-dominated

Before drawing any node-link picture, look at how degree is distributed
across all 41,073 nodes. This is the single most important structural
fact about this graph, and it explains everything the plots below will
show.


In [ ]:
degrees = [d for _, d in G.degree()]
n_total = len(degrees)
hub_degree = G.degree(hub)

print(f"Total nodes: {n_total}")
print(f"Busiest node ({hub}) degree: {hub_degree}  ->  {100*hub_degree/sum(degrees):.1f}% of all degree in the graph")
print(f"Nodes with degree == 1 (a single one-off conversation): {sum(1 for d in degrees if d==1)} ({100*sum(1 for d in degrees if d==1)/n_total:.1f}%)")
print(f"Nodes with degree >= 100: {sum(1 for d in degrees if d>=100)}")
print(f"Nodes with degree >= 10:  {sum(1 for d in degrees if d>=10)}")

top5 = sorted(G.degree, key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 busiest nodes:")
for node, d in top5:
    print(f"  {node}: degree={d}, touched_an_attack_flow={G.nodes[node]['is_malicious']}")


In [ ]:
plot_degree_distribution(G, title="Kaggle Graph - Degree Distribution (all 41,073 nodes, log-log)")


**What this shows:** almost all of the graph (95.8% of nodes) has degree
1 - a single one-off conversation with someone else. Degree then falls off
sharply, with only 27 nodes reaching degree 100 or higher, and one single
node - `147.32.84.229` - accounting for **30% of all edges in the entire
graph**. This is a classic single-vantage-point NetFlow signature: the
data was almost certainly captured from (or centered on) one monitored
host or gateway, so that host appears as a conversation partner in a huge
fraction of all recorded flows. It is not a data quality problem, and it
is not something graph sampling introduced - it is a structural property
of the raw data itself, confirmed here using the complete, un-sampled
41,073-node graph.

**Why this matters for the project:** with a structure this skewed, raw
degree centrality will rank the gateway node #1 by an enormous margin and
say almost nothing else useful - nearly every node will otherwise look
identical (degree 1). Betweenness, PageRank, and clustering coefficient
(computed later in this notebook) are more informative here precisely
*because* they respond differently to hub-and-spoke structure than raw
degree does.


## 4. What the hub neighborhood looks like

`plot_graph` automatically keeps only the 40 highest-degree nodes when a
graph is this large, and labels the plot to say so - never silently
showing a subset as if it were the whole thing.


In [ ]:
plot_graph(G, title=f"Hub neighborhood - busiest node {hub}", max_nodes=40)


## 5. Structure away from the main hub

`data/kaggle/` isn't *one* connected blob - it's a giant component plus many
small, separate fragments. Excluding the single dominant hub node reveals
the **second**-busiest node instead (still a strong hub, just an order of
magnitude smaller), which is worth seeing explicitly rather than assuming
the graph "flattens out" once the top node is removed.


In [ ]:
H = G.copy()
H.remove_node(hub)
second_hub, second_hub_degree = max(H.degree, key=lambda x: x[1])
print(f"With {hub} removed, the next busiest node is {second_hub} "
      f"- degree {second_hub_degree} in the remaining graph (unchanged from its original degree, since removing one other node barely affects it)")

G_no_hub = get_readable_subgraph(H, max_nodes=40)
plot_graph(G_no_hub, title=f"Same graph with the top hub ({hub}) removed", max_nodes=40)


## 6. Connected components: how many separate pieces does this graph have?

A graph can have a giant hub-dominated core AND many small, fully
disconnected fragments at the same time. Here's the exact breakdown.


In [ ]:
components = sorted(nx.connected_components(G), key=len, reverse=True)
sizes = [len(c) for c in components]

print(f"Total connected components: {len(components)}")
print(f"Giant component size: {sizes[0]} nodes ({100*sizes[0]/n_total:.1f}% of all nodes)")
print(f"Remaining {len(components)-1} components range from {min(sizes[1:])} to {max(sizes[1:])} nodes each")
print("Sizes of the 10 largest non-giant components:", sizes[1:11])


In [ ]:
small_component_nodes = sorted(components[1], key=str)
G_small = G.subgraph(components[1]).copy()
print(f"Showing one full non-giant component: {G_small.number_of_nodes()} nodes, {G_small.number_of_edges()} edges (no sampling needed - it's small enough to show completely)")

plot_graph(G_small, title="One isolated component, shown in full (not sampled)", max_nodes=G_small.number_of_nodes())


**Notice the shape repeats.** This small, fully separate 31-node
fragment is *also* a star: one node talking to 30 others who don't talk to
each other. The same one-host-to-many-peers pattern shows up at every
scale in this dataset - the giant hub, the second-tier hub, and even the
small disconnected fragments. That consistency is itself evidence this is
how the data was captured (from the perspective of individual hosts, each
one making many outbound connections), rather than a modeling artifact.


## 7. Where do the attacks sit structurally?

This is the question that actually matters for intrusion detection: are
malicious flows concentrated on the busiest nodes (easy to spot by degree
alone), or scattered among the quiet, one-off nodes (where degree tells
you nothing)?


In [ ]:
attack_edges = [(u, v) for u, v, d in G.edges(data=True) if float(d.get("ActivityLabel", 0)) != 0]
attack_nodes = set(n for edge in attack_edges for n in edge)

print(f"Attack-labeled edges: {len(attack_edges)} / {G.number_of_edges()} ({100*len(attack_edges)/G.number_of_edges():.2f}%)")
print(f"Distinct nodes touching at least one attack edge: {len(attack_nodes)}")
print(f"Is the #1 hub ({hub}) an attack node? {hub in attack_nodes}")

attack_degrees = sorted((G.degree(n) for n in attack_nodes), reverse=True)
print(f"\nAttack-node degree - top 10: {attack_degrees[:10]}")
print(f"Attack-node degree - median: {attack_degrees[len(attack_degrees)//2]}")
print(f"Attack nodes with degree == 1: {sum(1 for d in attack_degrees if d==1)} / {len(attack_degrees)}")

# do any of the small, non-giant components contain attack activity?
non_giant_with_attacks = sum(
    1 for c in components[1:]
    if any(float(d.get("ActivityLabel", 0)) != 0 for _, _, d in G.subgraph(c).edges(data=True))
)
print(f"\nNon-giant components containing an attack edge: {non_giant_with_attacks} / {len(components)-1}")


**Precise findings:**
- Only 2.63% of all edges are attack-labeled, and none of the top-5
  busiest nodes (including the main hub) touch a single one - the busiest
  node in this dataset is not implicated in any recorded attack.
- Attack-node degree is itself skewed: most attack-touching nodes only
  appear once (degree 1, same as the graph overall), but a handful reach
  degree in the hundreds - those are worth a closer look, since a node
  with 390 attack-flagged connections is a very different case from one
  with a single flagged connection.
- Every attack edge lives inside the giant component - none of the small,
  disconnected fragments contain any labeled attack activity.

**Takeaway for the team:** raw degree centrality alone would not have
flagged these attack nodes - they are structurally unremarkable (mostly
degree 1, same as 96% of the graph). This is a concrete, defensible reason
*why* the project looks beyond degree to betweenness, PageRank, and
clustering coefficient - and eventually to the ML/GNN models in Section 3
of the project.


In [ ]:
top_attack_nodes = sorted(attack_nodes, key=lambda n: G.degree(n), reverse=True)[:5]
print("Highest-degree attack-touching nodes:")
for n in top_attack_nodes:
    print(f"  {n}: degree={G.degree(n)}")

G_attack_view = get_readable_subgraph(G.subgraph(attack_nodes | {n for a in top_attack_nodes for n in G.neighbors(a)}), max_nodes=40)
plot_graph(G_attack_view, title="Neighborhood around the highest-degree attack-touching nodes", max_nodes=40)


## 8. Graph feature extraction

Computing exact betweenness centrality on all 41,073 nodes is expensive
(the exact algorithm is O(V·E)). We compute it - along with the other
four features - on the **300 highest-degree nodes**, which includes every
node that matters structurally (all hubs, all high-degree attack nodes)
while running in under a second. This is a deliberate, stated scope
choice, not a silent shortcut.


In [ ]:
G_features = get_readable_subgraph(G, max_nodes=300)
features = compute_all_features(G_features)
print(f"Computed features for {len(features)} nodes")
features.sort_values("betweenness_centrality", ascending=False).round(5).head(10)


In [ ]:
verify_features(G_features, features)


Both checks should read `True` - the handshake lemma (sum of degrees =
2x edges) and PageRank scores summing to ~1. If either is `False`, do not
trust the numbers above it.


In [ ]:
from visualize import plot_feature_distribution
plot_feature_distribution(features, "pagerank", title="PageRank distribution (top-300 subgraph)")


## Summary - what to say in the presentation

- This dataset is three views of the **same** 41,073-node, ~45K-edge
  graph: "plain" keeps one edge per IP pair (last flow wins), "aggregated"
  also keeps one edge per pair but combines repeated flows using an
  unspecified (not simple-average) rule, and "multi" keeps all 100,000
  original flows as separate parallel edges.
- The graph is **hub-dominated at every scale**: one node touches 30% of
  all edges, 95.8% of nodes have degree exactly 1, and even small,
  fully-disconnected fragments repeat the same one-to-many star shape.
  This is consistent with a single-vantage-point NetFlow capture.
- Attacks are **not** concentrated on the busiest nodes - the top 5 hubs,
  including the dominant one, touch zero attack-labeled edges. Attack
  nodes are mostly structurally unremarkable (degree 1), which is the
  concrete justification for using centrality features beyond raw degree,
  and eventually ML/GNN models, rather than a simple degree threshold.
- All numbers in this notebook come from the complete 41,073-node graph
  (no sampling) except the centrality features in Section 8, which are
  explicitly scoped to the top-300 nodes for compute-time reasons.
